In [6]:
import tensorflow as tf
import numpy as np

# --- Step 1: Importing necessary modules ---
# Already done above

# --- Step 2: Loading and preparing the MNIST data set ---
print("Loading data...")
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Flattening the 28x28 images into 784 features and normalizing (0 to 1)
x_train, x_test = x_train.reshape(-1, 784).astype('float32') / 255.0, \
                  x_test.reshape(-1, 784).astype('float32') / 255.0

# --- Step 3 & 4: Hyperparameters, Shuffling, and Batching ---
learning_rate = 0.1
training_steps = 1000
batch_size = 128
display_step = 100

train_data = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_data = train_data.repeat().shuffle(5000).batch(batch_size).prefetch(1)

# --- Step 5: Initializing weights and biases ---
# W shape: [784, 10] (784 pixels to 10 digit classes)
W = tf.Variable(tf.random.normal([784, 10]), name="weight")
b = tf.Variable(tf.zeros([10]), name="bias")

# --- Step 6: Defining logistic regression and cost function ---
def logistic_regression(x):
    # Apply softmax to normalize the logits to a probability distribution
    return tf.nn.softmax(tf.matmul(x, W) + b)

def cross_entropy(y_pred, y_true):
    # Encode label to a one hot vector
    y_true = tf.one_hot(y_true, depth=10)
    # Clip prediction values to avoid log(0) error
    y_pred = tf.clip_by_value(y_pred, 1e-9, 1.)
    # Compute cross-entropy
    return tf.reduce_mean(-tf.reduce_sum(y_true * tf.math.log(y_pred), axis=1))

# --- Step 7: Defining optimizers and accuracy metrics ---
optimizer = tf.optimizers.SGD(learning_rate)

def accuracy(y_pred, y_true):
    # Predicted class is the index of the highest score in prediction vector
    correct_prediction = tf.equal(tf.argmax(y_pred, 1), tf.cast(y_true, tf.int64))
    return tf.reduce_mean(tf.cast(correct_prediction, tf.float32))

# --- Step 8: Optimization process (Gradient Descent) ---
def run_optimization(x, y):
    with tf.GradientTape() as g:
        pred = logistic_regression(x)
        loss = cross_entropy(pred, y)
    
    # Compute gradients
    gradients = g.gradient(loss, [W, b])
    # Update W and b following gradients
    optimizer.apply_gradients(zip(gradients, [W, b]))

# --- Step 9: The training loop ---
print("\nStarting Training...")
for step, (batch_x, batch_y) in enumerate(train_data.take(training_steps), 1):
    run_optimization(batch_x, batch_y)
    
    if step % display_step == 0:
        pred = logistic_regression(batch_x)
        loss = cross_entropy(pred, batch_y)
        acc = accuracy(pred, batch_y)
        print(f"Step: {step}, Loss: {loss:.4f}, Training Accuracy: {acc:.4f}")

# --- Step 10: Testing model accuracy using test data ---
test_pred = logistic_regression(x_test)
print(f"\nFinal Test Accuracy: {accuracy(test_pred, y_test):.4f}")

Loading data...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step

Starting Training...
Step: 100, Loss: 3.9088, Training Accuracy: 0.3984
Step: 200, Loss: 2.1725, Training Accuracy: 0.5938
Step: 300, Loss: 1.6315, Training Accuracy: 0.6875
Step: 400, Loss: 1.3835, Training Accuracy: 0.7031
Step: 500, Loss: 1.1818, Training Accuracy: 0.7891
Step: 600, Loss: 1.6836, Training Accuracy: 0.7031
Step: 700, Loss: 1.3640, Training Accuracy: 0.7344
Step: 800, Loss: 1.6951, Training Accuracy: 0.7266
Step: 900, Loss: 0.7777, Training Accuracy: 0.8125
Step: 1000, Loss: 1.0477, Training Accuracy: 0.7656

Final Test Accuracy: 0.8034
